In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path
import time

dbalchemy = create_engine(f"postgresql+psycopg2://postgres@localhost:5432/dns_mac")

In [2]:
def get(day, hour0, hour1):
    query = f"""
    WITH 
    WINDOWS AS (
        SELECT id, dn_id, is_r, rcode, macsrc, macdst, FLOOR(SECONDS/3600) as hour FROM message3_it2016_0
        WHERE FLOOR(SECONDS/3600) BETWEEN {hour0} AND {hour1}
    ),
    TMP AS (
        SELECT
        {day} AS "day",
        hour AS "hour",
        dn.dac_family_rank1 AS DAC_FAMILY,
        
        COUNT(*) AS QR,
        COUNT(*) FILTER (WHERE IS_R IS FALSE) AS Q,
        COUNT(*) FILTER (WHERE RCODE = 0) AS OK,
        COUNT(*) FILTER (WHERE RCODE = 3) AS NX,
        COUNT(*) FILTER (WHERE NOT dn.regex_check) AS QR_NOTVALID,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check) AS Q_NOTVALID,
        COUNT(*) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check) AS NX_NOTVALID,
        COUNT(*) FILTER (WHERE RN=1) AS FA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1) AS FA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS FA_NX,
        COUNT(*) FILTER (WHERE RN_MAC=1) AS MACFA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1) AS MACFA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS MACFA_NX,


        COUNT(DISTINCT DN_ID) FILTER (WHERE P1) AS  P1_DN,
        COUNT(DISTINCT DN_ID) FILTER (WHERE NOT dn.regex_check AND P1) AS  P1_DN_NOTVALID,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=0 AND P1) AS  P1_DN_OK,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND P1) AS  P1_DN_NXD,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND NOT dn.regex_check AND P1) AS  P1_DN_NXD_NOTVALID,
        COUNT(*) FILTER (WHERE P1) AS  P1_QR,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND P1) AS  P1_Q,
        COUNT(*) FILTER (WHERE RCODE = 0 AND P1) AS  P1_OK,
        COUNT(*) FILTER (WHERE RCODE = 3 AND P1) AS  P1_NX,
        COUNT(*) FILTER (WHERE NOT dn.regex_check AND P1) AS  P1_QR_NOTVALID,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check AND P1) AS  P1_Q_NOTVALID,
        COUNT(*) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check AND P1) AS  P1_NX_NOTVALID,
        COUNT(*) FILTER (WHERE RN=1 AND P1) AS  P1_FA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1 AND P1) AS  P1_FA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1 AND P1) AS  P1_FA_NX,
        COUNT(*) FILTER (WHERE RN_MAC=1 AND P1) AS  P1_MACFA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1 AND P1) AS  P1_MACFA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1 AND P1) AS  P1_MACFA_NX,

        SUM(LOGIt1) AS LLR1_QR,
        SUM(LOGIt1) FILTER (WHERE IS_R IS FALSE) AS LLR1_Q,
        SUM(LOGIt1) FILTER (WHERE RCODE = 0) AS LLR1_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE = 3) AS LLR1_NX,
        SUM(LOGIt1) FILTER (WHERE NOT dn.regex_check) AS LLR1_QR_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check) AS LLR1_Q_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check) AS LLR1_NX_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE RN=1) AS LLR1_FA,
        SUM(LOGIt1) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1) AS LLR1_FA_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS LLR1_FA_NX,
        SUM(LOGIt1) FILTER (WHERE RN_MAC=1) AS LLR1_MACFA,
        SUM(LOGIt1) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1) AS LLR1_MACFA_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS LLR1_MACFA_NX

        FROM WINDOWS M
        JOIN bigdn_m3_3 DN ON M.DN_ID=DN.ID
        JOIN M3_RN ON M3_RN.id=M.ID
        GROUP BY "day", hour, DAC_FAMILY
    )
    SELECT tmp.* FROM tmp
    """
    dfright = pd.read_sql(query, dbalchemy)
    # if Path('/tmp/dfright.csv').exists():
    #     dfright = pd.read_csv('/tmp/dfright.csv', index_col=0)
    # else:
    #     dfright = pd.read_sql(query, dbalchemy)
    #     dfright.to_csv('/tmp/dfright.csv')
    #     pass
    dflefts = []
    for idx, row in dfright.iterrows():
        dflefts.append(pd.read_sql(f"""select * from get_hourdn_statistics('{row['dac_family']}', {row['hour']}::int)""", dbalchemy))
    return pd.concat([pd.concat(dflefts).reset_index(drop=True), dfright.drop(columns='dac_family')], axis=1)

In [ ]:

hstep = 2
for day in range(10):
    dfs = []
    for h in range(0, 24 - hstep, hstep):
        s = time.time()
        h0, h1 = h, h + hstep - 1
        print(day, h0, h1, end=' ')
        fn = f'hourly/day{day}_{h0}_{h1}.csv'
        if Path(fn).exists():
            df = pd.read_csv(fn, index_col=0)
        else:
            df = get(day, h, h1) # BETWEEN is INCLUSIVE
        df.to_csv(fn)
        dfs.append(df)
        print(f'{time.time() - s}s')
        pass
    pd.concat(dfs).to_csv(f'daily/day{day}.csv')
    pass


0 0 1 31.076178073883057s
0 2 3 17.586993932724s
0 4 5 8.632561922073364s
0 6 7 8.32303500175476s
0 8 9 19.603675842285156s
0 10 11 38.9787540435791s
0 12 13 52.820175886154175s
0 14 15 55.51001191139221s
0 16 17 41.3282687664032s
0 18 19 36.73177194595337s
0 20 21 40.296513080596924s
1 0 1 30.35823893547058s
1 2 3 16.622484922409058s
1 4 5 8.509649276733398s
1 6 7 8.195676803588867s
1 8 9 17.107188940048218s
1 10 11 46.75485110282898s
1 12 13 37.51571989059448s
1 14 15 43.121570110321045s
1 16 17 35.261739015579224s
1 18 19 37.80721712112427s
1 20 21 38.478474855422974s
2 0 1 32.49740505218506s
2 2 3 17.561856985092163s
2 4 5 7.968089818954468s
2 6 7 8.075881958007812s
2 8 9 18.002470016479492s
2 10 11 34.35684299468994s
2 12 13 38.73911190032959s
2 14 15 44.61906623840332s
2 16 17 38.352092027664185s
2 18 19 36.53742790222168s
2 20 21 41.50281500816345s
3 0 1 29.821730136871338s
3 2 3 17.692699909210205s
3 4 5 8.026705026626587s
3 6 7 8.527643203735352s
3 8 9 18.88170099258423s
3 10 